# Lumenary — 3D Gaussian Splatting Training

Trains a 3DGS model on a free T4 GPU (~45-60 min including COLMAP).

**CRITICAL: Runtime > Change runtime type > T4 GPU**

Set the scene name in Step 0, then run all cells in order.

In [ ]:
#@title ## Step 0 — Configuration
#@markdown Set which scene you're training. Run this cell FIRST.

SCENE_NAME = "okavango_delta"  #@param ["okavango_delta", "gaborone_city"]

#@markdown Max image dimension (pixels). Lower = less VRAM, faster.
#@markdown 1600 is safe for T4 (16GB). 1200 is safer.
MAX_DIM = 1600  #@param [1200, 1600, 1920]

#@markdown Subsample rate. 1=every frame, 2=every other, 3=every 3rd.
#@markdown Video frames have high temporal overlap. Use 3 for best results.
SUBSAMPLE_RATE = 3  #@param [1, 2, 3, 4, 5]

#@markdown Training iterations. 30000 is standard. 15000 is faster but lower quality.
ITERATIONS = 30000  #@param [15000, 20000, 30000]

print(f"Scene: {SCENE_NAME}")
print(f"Max dimension: {MAX_DIM}px")
print(f"Subsample rate: every {SUBSAMPLE_RATE} frames")
print(f"Training iterations: {ITERATIONS}")

## Step 1 — Install Dependencies

~3-5 minutes. Installs PyTorch (matching Colab's CUDA), COLMAP, and 3DGS.

In [ ]:
#@title ## Step 1 — Install everything (run once)
import subprocess, os, sys, re

# --- 1a. Detect Colab's system CUDA version ---
def get_system_cuda():
    """Get CUDA version from nvidia-smi or nvcc."""
    try:
        out = subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT, text=True)
        # nvidia-smi shows CUDA Version: X.Y
        m = re.search(r'CUDA Version:\s+(\d+)\.(\d+)', out)
        if m:
            return f"{m.group(1)}.{m.group(2)}"
    except Exception:
        pass
    try:
        out = subprocess.check_output(['nvcc', '--version'], stderr=subprocess.STDOUT, text=True)
        m = re.search(r'release (\d+)\.(\d+)', out)
        if m:
            return f"{m.group(1)}.{m.group(2)}"
    except Exception:
        pass
    return None

system_cuda = get_system_cuda()
print(f"System CUDA version: {system_cuda}")

# Map CUDA version to PyTorch wheel tag
# Note: cu130 doesn't exist in PyTorch wheels, fall back to cu128
cuda_to_wheel = {
    "11.8": "cu118",
    "12.1": "cu121",
    "12.4": "cu124",
    "12.6": "cu126",
    "12.8": "cu128",
}

# Find closest matching wheel
if system_cuda:
    major_minor = ".".join(system_cuda.split('.')[:2])
    if major_minor in cuda_to_wheel:
        torch_cuda = cuda_to_wheel[major_minor]
    elif system_cuda.startswith("12."):
        torch_cuda = "cu126"  # fallback for CUDA 12.x
    elif system_cuda.startswith("13."):
        torch_cuda = "cu128"  # cu130 not available, cu128 is compatible
    else:
        torch_cuda = "cu126"  # safe default
else:
    torch_cuda = "cu126"  # safe default if detection fails

print(f"Installing PyTorch with {torch_cuda} wheels")

# --- 1b. Install system packages ---
!apt-get update -qq && apt-get install -y -qq ninja-build colmap libglm-dev > /dev/null 2>&1

# --- 1c. Install PyTorch (matching system CUDA) ---
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/{torch_cuda}

# --- 1d. Install pip dependencies ---
!pip install -q plyfile Pillow tqdm opencv-python joblib

# --- 1e. Clone 3DGS repo ---
if not os.path.exists('/content/3dgs'):
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting /content/3dgs

# --- 1f. cd into repo ---
%cd /content/3dgs

# --- 1g. Install 3DGS submodules ---
import torch as _torch
_cc = _torch.cuda.get_device_capability(0) if _torch.cuda.is_available() else (7, 5)
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{_cc[0]}.{_cc[1]}"
print(f"GPU compute capability: {_cc[0]}.{_cc[1]}")

!pip install --no-build-isolation -q submodules/diff-gaussian-rasterization
!pip install --no-build-isolation -q submodules/simple-knn
!pip install --no-build-isolation -q submodules/fused-ssim

# --- 1h. Headless rendering ---
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'

# --- 1i. Verify ---
import torch
print(f"\n{'='*50}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"PyTorch CUDA: {torch.version.cuda}")

# Verify diff-gaussian-rasterization installed
try:
    import diff_gaussian_rasterization
    print("diff-gaussian-rasterization: OK")
except ImportError as e:
    print(f"ERROR: diff-gaussian-rasterization not installed: {e}")
    print("Try restarting runtime and re-running this cell.")

print(f"{'='*50}")
print("Installation complete!")

## Step 2 — Upload Training Images

Upload a zip file containing the extracted video frames.

Frames are at:
- `processing/okavango_delta/frames/*.jpg` (52 frames)
- `processing/gaborone_city/frames/*.jpg` (20 frames)

To zip locally: `cd processing && zip -r /tmp/{scene}_frames.zip {scene}_frames/`

In [ ]:
#@title ## Step 2 — Upload and prepare images
import os, zipfile, glob, shutil, hashlib, sys
from PIL import Image

# Ensure Step 0 was run (provide defaults if not)
if 'SCENE_NAME' not in dir():
    SCENE_NAME = "okavango_delta"
    print(f"Warning: Step 0 not run. Using default SCENE_NAME={SCENE_NAME}")
if 'SUBSAMPLE_RATE' not in dir():
    SUBSAMPLE_RATE = 3
if 'MAX_DIM' not in dir():
    MAX_DIM = 1600
if 'ITERATIONS' not in dir():
    ITERATIONS = 30000
if 'image_exts' not in dir():
    image_exts = ('.jpg', '.jpeg', '.png', '.webp')

INPUT_DIR = "/content/input_images_raw"
PREPARED_DIR = "/content/input_images"

# Clear stale data from previous runs
for _d in [INPUT_DIR, PREPARED_DIR]:
    if os.path.exists(_d): shutil.rmtree(_d)
    os.makedirs(_d, exist_ok=True)
os.makedirs(PREPARED_DIR, exist_ok=True)

# Upload
from google.colab import files
print(f"Upload frames zip for {SCENE_NAME}:")
print(f"(Must be a .zip file containing .jpg images, e.g. okavango_delta_frames.zip)")
uploaded = files.upload()

if not uploaded:
    print("ERROR: No file uploaded. Re-run this cell and select the frames zip.")
    sys.exit(1)

for filename, data in uploaded.items():
    filepath = os.path.join(INPUT_DIR, filename)
    with open(filepath, "wb") as f:
        f.write(data)
    if filename.endswith('.zip'):
        print(f"  Extracting {filename}...")
        with zipfile.ZipFile(filepath, 'r') as z:
            z.extractall(INPUT_DIR)
        os.remove(filepath)
    else:
        print(f"  WARNING: '{filename}' is not a .zip file.")
        print(f"  You uploaded a non-zip file. Please re-run and upload the correct zip.")
        sys.exit(1)

# Find all images (handle nested dirs from zip)
image_exts = ('.jpg', '.jpeg', '.png', '.webp')
all_images = []
for root, dirs, files_list in os.walk(INPUT_DIR):
    for f in files_list:
        if f.lower().endswith(image_exts):
            all_images.append(os.path.join(root, f))

if not all_images:
    print(f"ERROR: No images found in the uploaded file.")
    print(f"  Make sure your zip contains .jpg images, not another file.")
    print(f"  Expected: okavango_delta_frames.zip containing frame_0001.jpg, frame_0002.jpg, etc.")
    sys.exit(1)

# Flatten into PREPARED_DIR
for img_path in all_images:
    dst = os.path.join(PREPARED_DIR, os.path.basename(img_path))
    shutil.copy2(img_path, dst)

# Sort and get frame list
frames = sorted([f for f in os.listdir(PREPARED_DIR) if f.lower().endswith(image_exts)])
print(f"\nTotal images found: {len(frames)}")

# --- Filter: remove duplicate content (by file hash) ---
seen_hashes = set()
unique_frames = []
for f in frames:
    path = os.path.join(PREPARED_DIR, f)
    h = hashlib.md5(open(path, 'rb').read()).hexdigest()
    if h not in seen_hashes:
        seen_hashes.add(h)
        unique_frames.append(f)

removed_dupes = len(frames) - len(unique_frames)
if removed_dupes > 0:
    print(f"Removed {removed_dupes} duplicate frames")
frames = unique_frames

# --- Filter: remove low-quality frames (very small file = blurry/simple) ---
if len(frames) > 0:
    sizes = [(f, os.path.getsize(os.path.join(PREPARED_DIR, f))) for f in frames]
    avg_size = sum(s for _, s in sizes) / len(sizes)
    quality_frames = [f for f, s in sizes if s > avg_size * 0.4]  # remove < 40% of avg
    removed_blur = len(frames) - len(quality_frames)
    if removed_blur > 0:
        print(f"Removed {removed_blur} low-quality/blurry frames (file size < 40% avg)")
    frames = quality_frames

if len(frames) == 0:
    print(f"ERROR: No valid images remaining after filtering.")
    print(f"  Re-run this cell and upload the correct frames zip.")
    sys.exit(1)

# --- Subsample for temporal diversity ---
if SUBSAMPLE_RATE > 1:
    subsampled = frames[::SUBSAMPLE_RATE]
    print(f"Subsampled: {len(frames)} -> {len(subsampled)} frames (every {SUBSAMPLE_RATE}th frame)")
    frames = subsampled

# --- Resize for T4 VRAM ---
print(f"\nResizing images (max {MAX_DIM}px on longest side)...")
resized_count = 0
for f in frames:
    src = os.path.join(PREPARED_DIR, f)
    img = Image.open(src)
    w, h = img.size
    max_side = max(w, h)
    if max_side > MAX_DIM:
        ratio = MAX_DIM / max_side
        new_w, new_h = int(w * ratio), int(h * ratio)
        img = img.resize((new_w, new_h), Image.LANCZOS)
        img.save(src, quality=95)
        resized_count += 1

if resized_count > 0:
    print(f"Resized {resized_count} images")

# --- Final report ---
print(f"\n{'='*50}")
print(f"FINAL: {len(frames)} images ready for training")
if len(frames) < 10:
    print("WARNING: Fewer than 10 images. COLMAP may fail or produce poor results.")
    print("Consider using more source video or lowering SUBSAMPLE_RATE.")
elif len(frames) < 20:
    print("NOTE: 10-20 images. Results may be limited. Consider more source video.")
elif len(frames) > 80:
    print("NOTE: >80 images. Training will be slower. Consider increasing SUBSAMPLE_RATE.")

# Show sample image info
sample = Image.open(os.path.join(PREPARED_DIR, frames[0]))
print(f"Sample image: {frames[0]} ({sample.size[0]}x{sample.size[1]})")
print(f"{'='*50}")

## Step 3 — Preview Images

Visual check before running COLMAP. Look for:
- Enough visual diversity (different viewpoints)
- No obviously blurry/dark images
- Consistent exposure

In [ ]:
#@title ## Step 3 — Preview images (run this to visually check)
from IPython.display import display, HTML
from PIL import Image
import os

# Ensure earlier steps were run
if 'PREPARED_DIR' not in dir():
    PREPARED_DIR = '/content/input_images'
if 'image_exts' not in dir():
    image_exts = ('.jpg', '.jpeg', '.png', '.webp')

frames = sorted([f for f in os.listdir(PREPARED_DIR) if f.lower().endswith(image_exts)])

# Show grid of thumbnails
html = "<div style='display:flex;flex-wrap:wrap;gap:4px'>"
for f in frames[:30]:  # show first 30
    path = os.path.join(PREPARED_DIR, f)
    img = Image.open(path)
    img.thumbnail((150, 150))
    # Save as temp for display
    tmp = f"/tmp/thumb_{f}"
    img.save(tmp)
    import base64
    with open(tmp, 'rb') as fh:
        b64 = base64.b64encode(fh.read()).decode()
    html += f"<img src='data:image/jpeg;base64,{b64}' style='height:120px;margin:2px'>"
html += "</div>"
display(HTML(html))

print(f"\nShowing {min(30, len(frames))} of {len(frames)} images")
print("Check: Are there different viewpoints? Any obviously bad images?")
print("If images look too similar, increase SUBSAMPLE_RATE in Step 0 and re-run from Step 2.")

## Step 4 — Run COLMAP (Camera Pose Estimation)

Estimates camera positions for each frame.
~5-15 minutes depending on image count.

In [ ]:
#@title ## Step 4 — Run COLMAP
import subprocess, os, shutil

# Ensure earlier steps were run
if 'PREPARED_DIR' not in dir():
    PREPARED_DIR = '/content/input_images'
if 'image_exts' not in dir():
    image_exts = ('.jpg', '.jpeg', '.png', '.webp')

os.environ['QT_QPA_PLATFORM'] = 'offscreen'

COLMAP_workspace = "/content/colmap_workspace"
COLMAP_database = os.path.join(COLMAP_workspace, "database.db")
COLMAP_images = os.path.join(COLMAP_workspace, "images")
COLMAP_sparse = os.path.join(COLMAP_workspace, "sparse")

os.makedirs(COLMAP_workspace, exist_ok=True)
os.makedirs(COLMAP_sparse, exist_ok=True)

# Clean previous run
if os.path.exists(COLMAP_database):
    os.remove(COLMAP_database)
if os.path.islink(COLMAP_images):
    os.unlink(COLMAP_images)
elif os.path.isdir(COLMAP_images):
    shutil.rmtree(COLMAP_images)

# Symlink prepared images
os.symlink(PREPARED_DIR, COLMAP_images)

n_images = len([f for f in os.listdir(COLMAP_images) if f.lower().endswith(image_exts)])
print(f"Running COLMAP on {n_images} images...")

# Step 4a: Feature extraction (CPU)
print("\n[1/3] Extracting features (CPU, ~2-5 min)...")
!QT_QPA_PLATFORM=offscreen colmap feature_extractor \
    --database_path {COLMAP_database} \
    --image_path {COLMAP_images} \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 0

print("\n[1/3] Feature extraction complete.")

# Step 4b: Feature matching (CPU)
print("\n[2/3] Matching features (CPU, ~2-10 min)...")
!QT_QPA_PLATFORM=offscreen colmap exhaustive_matcher \
    --database_path {COLMAP_database} \
    --SiftMatching.use_gpu 0

print("\n[2/3] Feature matching complete.")

# Step 4c: Sparse reconstruction
print("\n[3/3] Running mapper (~1-5 min)...")
!QT_QPA_PLATFORM=offscreen colmap mapper \
    --database_path {COLMAP_database} \
    --image_path {COLMAP_images} \
    --output_path {COLMAP_sparse}

# Verify
sparse_dir = os.path.join(COLMAP_sparse, "0")
if os.path.exists(sparse_dir):
    files = os.listdir(sparse_dir)
    print(f"\n{'='*50}")
    print(f"COLMAP reconstruction successful!")
    print(f"Output files: {files}")
    
    # Check how many cameras were registered
    import struct
    images_bin = os.path.join(sparse_dir, 'images.bin')
    if os.path.exists(images_bin):
        with open(images_bin, 'rb') as f:
            num_images = struct.unpack('Q', f.read(8))[0]
            print(f"Cameras registered: {num_images} / {n_images}")
            if num_images < n_images * 0.7:
                print(f"WARNING: Only {num_images}/{n_images} images registered.")
                print("This means COLMAP couldn't match enough features.")
                print("Causes: images too similar, blurry, or insufficient overlap.")
                print("Fix: Increase SUBSAMPLE_RATE in Step 0 and re-run from Step 2.")
    print(f"{'='*50}")
else:
    print(f"\nERROR: COLMAP failed. No reconstruction in {COLMAP_sparse}")
    print("Common fixes:")
    print("  1. Increase SUBSAMPLE_RATE (images too similar)")
    print("  2. Check image quality in Step 3 (too blurry)")
    print("  3. Ensure images have sufficient overlap")

## Step 5 — Train the 3DGS Model

30K iterations on T4 GPU: ~30-45 minutes.

In [ ]:
#@title ## Step 5 — Train
import torch

# Ensure earlier steps were run
if 'ITERATIONS' not in dir():
    ITERATIONS = 30000
COLMAP_workspace = '/content/colmap_workspace'

# VRAM check
if torch.cuda.is_available():
    free_mem = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f"Available VRAM: {free_mem / 1e9:.1f} GB")
    if free_mem < 8e9:
        print("WARNING: Less than 8GB VRAM available. Training may fail.")
        print("Try restarting runtime or reducing MAX_DIM in Step 0.")

%cd /content/3dgs

!python train.py \
    -s {COLMAP_workspace} \
    -m /content/output_model \
    --iterations {ITERATIONS}

print("\nTraining complete!")

## Step 6 — Export the PLY File

In [ ]:
#@title ## Step 6 — Export PLY
import os

# Ensure earlier steps were run
if 'ITERATIONS' not in dir():
    ITERATIONS = 30000
if 'SCENE_NAME' not in dir():
    SCENE_NAME = 'okavango_delta'

PLY_PATH = f"/content/output_model/point_cloud/iteration_{ITERATIONS}/point_cloud.ply"

if os.path.exists(PLY_PATH):
    size_mb = os.path.getsize(PLY_PATH) / (1024 * 1024)
    print(f"PLY file ready: {PLY_PATH}")
    print(f"File size: {size_mb:.1f} MB")
    print(f"\nDownload this file, then upload to GCS:")
    print(f"  gsutil cp {SCENE_NAME}_point_cloud.ply gs://lumenary-viewer-us/scenes/{SCENE_NAME}/scene.ply")
else:
    print(f"ERROR: PLY not found at {PLY_PATH}")
    print("Searching for PLY files...")
    for root, dirs, files in os.walk("/content/output_model"):
        for f in files:
            if f.endswith('.ply'):
                print(f"  Found: {os.path.join(root, f)}")

In [ ]:
#@title ## Step 7 — Download PLY to your machine
from google.colab import files
import os

# Ensure earlier steps were run
if 'SCENE_NAME' not in dir():
    SCENE_NAME = 'okavango_delta'
if 'ITERATIONS' not in dir():
    ITERATIONS = 30000
PLY_PATH = f"/content/output_model/point_cloud/iteration_{ITERATIONS}/point_cloud.ply"

output_name = f"{SCENE_NAME}_point_cloud.ply"
os.rename(PLY_PATH, f"/content/{output_name}")

files.download(f"/content/{output_name}")
print(f"Downloading {output_name}...")

print(f"\nAfter download, upload to GCS:")
print(f"  gsutil cp {output_name} gs://lumenary-viewer-us/scenes/{SCENE_NAME}/scene.ply")
print(f"  gsutil iam ch allUsers:objectViewer gs://lumenary-viewer-us")
print(f"\nThen refresh http://localhost:3000 and enter the scene!")